In [1]:
import subprocess, sys

print(" Đang cài đặt thư viện...")

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.40.0",
    "torch",
    "gradio>=4.0.0",
    "openai-whisper",
    "nltk",
    "ipywidgets",
    "ffmpeg-python",
    "rouge-score"
], check=True)

subprocess.run(["apt-get", "install", "-y", "-q", "ffmpeg"], check=False)

import torch
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n Cài xong!")
print(f" Device: {DEVICE}")

 Đang cài đặt thư viện...

 Cài xong!
 Device: cuda


In [2]:
from google.colab import drive
import os, zipfile

drive.mount("/content/drive")

BART_MODEL_PATH = "/content/bart_best"
ZIP_ON_DRIVE    = "/content/drive/MyDrive/NLP_CK/bart_best.zip"

if os.path.exists(BART_MODEL_PATH) and os.listdir(BART_MODEL_PATH):
    print("✅ Model đã có sẵn!")
else:
    print("📦 Đang giải nén model... (~30-60 giây)")
    with zipfile.ZipFile(ZIP_ON_DRIVE, "r") as zf:
        zf.extractall("/content/")
    print("✅ Giải nén xong!")

files_found = os.listdir(BART_MODEL_PATH)
required = ["config.json", "model.safetensors", "tokenizer.json"]
print("\nKiểm tra files:")
for req in required:
    print(f"  {'✅' if req in files_found else '❌'} {req}")

Mounted at /content/drive
📦 Đang giải nén model... (~30-60 giây)
✅ Giải nén xong!

Kiểm tra files:
  ✅ config.json
  ✅ model.safetensors
  ✅ tokenizer.json


In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

print(f"⏳ Đang load BART model từ: {BART_MODEL_PATH}")

bart_tokenizer = AutoTokenizer.from_pretrained(BART_MODEL_PATH)

bart_model = AutoModelForSeq2SeqLM.from_pretrained(
    BART_MODEL_PATH,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)

bart_model.eval()

n_params = sum(p.numel() for p in bart_model.parameters()) / 1e6
print(f"\n✅ BART model loaded thành công!")
print(f"   Số tham số: {n_params:.0f}M")
print(f"   Device: {DEVICE}")

⏳ Đang load BART model từ: /content/bart_best


`torch_dtype` is deprecated! Use `dtype` instead!
Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]


✅ BART model loaded thành công!
   Số tham số: 406M
   Device: cuda


In [4]:
import re, time, os
from collections import Counter
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords
from rouge_score import rouge_scorer as _rouge_scorer

_ROUGE = _rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

try:
    STOP_WORDS = set(stopwords.words("english"))
except:
    STOP_WORDS = {"the","a","an","is","it","in","of","and","to","for","with","was","are","be","this","that"}

# ====================== TEXT CLEANING ======================
def clean_text(text):
    text = re.sub(r"\d{2}:\d{2}:\d{2}[,.]\d{3}\s*-->\s*\d{2}:\d{2}:\d{2}[,.]\d{3}", "", text)
    text = re.sub(r"^\d+\s*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"[\r\n]+", " ", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip()

# ====================== KEYWORD EXTRACTION ======================
def extract_keywords(text, top_k=12):
    words = [w.lower() for w in text.split() if w.isalpha() and w.lower() not in STOP_WORDS and len(w) > 3]
    freq = Counter(words)
    top = [w for w, _ in freq.most_common(top_k)]
    return ", ".join(top) if top else "(không tìm được từ khóa)"

# ====================== LEAD-3 ======================
def lead3_summarize(text):
    sentences = sent_tokenize(text)
    return " ".join(sentences[:3]) if sentences else text[:500]

# ====================== BART SUMMARIZE ======================
def bart_summarize(transcript, max_output=256, min_output=60, num_beams=4):
    """
    Inference đúng với cách fine-tune:
    - truncation tại 1024 token (khớp với training config)
    - num_beams=4, length_penalty=2.0 cho abstractive output
    """
    t0 = time.time()
    inputs = bart_tokenizer(
        transcript,
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        output_ids = bart_model.generate(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_length=max_output,
            min_length=min_output,
            num_beams=num_beams,
            length_penalty=2.0,
            no_repeat_ngram_size=3,
            repetition_penalty=1.2,
            early_stopping=True,
            forced_bos_token_id=0,
        )
    summary = bart_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    return summary, time.time() - t0

# ====================== EVALUATE ======================
def evaluate_summary(transcript: str, bart_s: str, lead3: str):
    words_t = transcript.lower().split()
    words_s = bart_s.lower().split()
    words_l = lead3.lower().split()
    n_t, n_s = len(words_t), len(words_s)

    # Compression ratio
    ratio = n_s / max(n_t, 1)

    # ROUGE: BART vs transcript (đo độ bao phủ nội dung gốc)
    r_vs_trans = _ROUGE.score(transcript[:2000], bart_s)

    # ROUGE: Lead-3 vs transcript (để so sánh baseline)
    r_lead_vs_trans = _ROUGE.score(transcript[:2000], lead3)

    # Abstractive degree: % unigram trong BART KHÔNG xuất hiện nguyên văn trong Lead-3
    set_lead = set(words_l)
    non_extractive = [w for w in words_s if w not in set_lead and w.isalpha() and len(w) > 2]
    abstractive_ratio = len(non_extractive) / max(n_s, 1) * 100

    # Sentence-level novelty: câu BART không copy nguyên câu Lead-3
    bart_sents = [s.strip().lower() for s in bart_s.split(".") if s.strip()]
    lead_sents = [s.strip().lower() for s in lead3.split(".") if s.strip()]
    novel_sents = sum(1 for s in bart_sents if not any(s in l or l in s for l in lead_sents))
    sent_novelty = novel_sents / max(len(bart_sents), 1) * 100

    comment = f"""📊 Phân tích Output
{'─'*50}
📏 Độ nén
  • Input          : {n_t:,} từ
  • BART output    : {n_s:,} từ
  • Tỉ lệ nén      : {ratio*100:.1f}%  (benchmark VT-SSum: ~11%)

📐 ROUGE vs Transcript gốc
  • BART  ROUGE-1  : {r_vs_trans['rouge1'].fmeasure*100:.1f}%
  • BART  ROUGE-L  : {r_vs_trans['rougeL'].fmeasure*100:.1f}%
  • Lead-3 ROUGE-1 : {r_lead_vs_trans['rouge1'].fmeasure*100:.1f}%  (baseline extractive)
  • Lead-3 ROUGE-L : {r_lead_vs_trans['rougeL'].fmeasure*100:.1f}%  (baseline extractive)

🔬 Tính Abstractive (so với Lead-3)
  • Từ khác Lead-3 : {abstractive_ratio:.1f}%  ← BART diễn đạt khác baseline
  • Câu mới hoàn toàn: {sent_novelty:.1f}%  ← không copy nguyên câu Lead-3
{'─'*50}
ℹ️  Kết quả trên tập test (400 mẫu — VT-SSum):
  ROUGE-1=42.16% | ROUGE-2=21.62% | ROUGE-L=24.98%
  METEOR=26.19%  | BERTScore-F1=0.8555"""

    return comment

# ====================== PIPELINE ======================
def run_pipeline_text(raw_text, max_output=256, min_output=60, num_beams=4):
    if not raw_text or not raw_text.strip():
        return ("", "", "⚠️ Vui lòng nhập transcript!", "", "")
    cleaned = clean_text(raw_text)
    word_count = len(cleaned.split())
    if word_count < 20:
        return (cleaned, "", f"⚠️ Transcript quá ngắn ({word_count} từ).", "", "")

    keywords = extract_keywords(cleaned)
    lead3    = lead3_summarize(cleaned)
    bart_s, t = bart_summarize(cleaned, max_output, min_output, num_beams)
    eval_comment = evaluate_summary(cleaned, bart_s, lead3)

    stats = f"{eval_comment}\n\n⏱️ Thời gian sinh: {t:.2f}s | Device: {DEVICE.upper()}"
    preview = cleaned[:2000] + ("..." if len(cleaned) > 2000 else "")
    return (preview, keywords, bart_s, lead3, stats)

# ====================== WHISPER ======================
WHISPER_MODEL = None
WHISPER_MODEL_SIZE = "base"

def load_whisper(size="base"):
    global WHISPER_MODEL, WHISPER_MODEL_SIZE
    if WHISPER_MODEL is None or size != WHISPER_MODEL_SIZE:
        import whisper
        print(f"⏳ Loading Whisper {size}...")
        WHISPER_MODEL = whisper.load_model(size)
        WHISPER_MODEL_SIZE = size
        print(f"✅ Whisper {size} loaded!")
    return WHISPER_MODEL

def run_pipeline_video(video_path, max_output=256, min_output=60, num_beams=4, whisper_size="base"):
    if video_path is None:
        return ("", "", "⚠️ Chưa upload video!", "", "")
    try:
        wm = load_whisper(whisper_size)
        t_start = time.time()
        file_size_mb = os.path.getsize(video_path) / (1024 * 1024)
        print(f"📦 Video size: {file_size_mb:.1f} MB")
        result = wm.transcribe(video_path, fp16=False, verbose=False,
                               condition_on_previous_text=False,
                               temperature=0, beam_size=1, best_of=1, language=None)
        transcript = result["text"]
        lang = result.get("language", "?")
        asr_time = time.time() - t_start
        print(f"✅ Transcribed {len(transcript.split())} từ ({asr_time:.1f}s)")
        preview, keywords, bart_s, lead3_s, stats = run_pipeline_text(transcript, max_output, min_output, num_beams)
        stats = f"🎙️ ASR: {asr_time:.1f}s | Ngôn ngữ: {lang} | File: {file_size_mb:.1f}MB\n" + stats
        return (preview, keywords, bart_s, lead3_s, stats)
    except Exception as e:
        return ("", "", f"❌ Lỗi: {str(e)}", "", "")

print("✅ Pipeline loaded — BART fine-tuned on VT-SSum | Abstractive Summarization")


✅ Pipeline loaded — BART fine-tuned on VT-SSum | Abstractive Summarization


In [ ]:
import gradio as gr

def gradio_text(text, max_tokens, min_tokens, num_beams):
    return run_pipeline_text(text, max_output=max_tokens, min_output=min_tokens, num_beams=num_beams)

def gradio_srt(file_obj):
    if file_obj is None:
        return ("", "", "⚠️ Chưa upload file!", "", "")
    try:
        raw = open(file_obj.name, encoding="utf-8", errors="ignore").read()
        return gradio_text(raw, 300, 80, 6)
    except Exception as e:
        return ("", "", f"❌ Lỗi: {e}", "", "")

def gradio_video(video_path, max_tokens, min_tokens, num_beams, whisper_size="base"):
    global WHISPER_MODEL, WHISPER_MODEL_SIZE
    if whisper_size != WHISPER_MODEL_SIZE:
        WHISPER_MODEL = None
    WHISPER_MODEL_SIZE = whisper_size
    return run_pipeline_video(video_path, max_tokens, min_tokens, num_beams, whisper_size)

# CSS tùy chỉnh
CSS = """
#bart-output textarea { background:#f0fff4 !important; border-left:4px solid #38a169 !important; font-size: 15px; }
#lead3-output textarea { background:#fffaf0 !important; border-left:4px solid #dd6b20 !important; }
#stats-output textarea { background:#ebf8ff !important; border-left:4px solid #3182ce !important; }
"""

with gr.Blocks(title="🎥 Video Summarization - Topic 17", theme=gr.themes.Soft(primary_hue="purple"), css=CSS) as demo:

    gr.Markdown("""
    # 🎥 Video Subtitle Understanding & Summarization
    **NLP Project – Topic 17 · IUH 2026**
    Mô hình: `BART-large-cnn` fine-tuned trên VT-SSum
    """)

    with gr.Tabs():

        # Tab 1: Nhập Transcript
        with gr.Tab("📝 Nhập Transcript"):
            with gr.Row():
                with gr.Column(scale=3):
                    txt_in = gr.Textbox(
                        label="📄 Transcript đầu vào",
                        lines=12,
                        placeholder="Dán transcript hoặc phụ đề vào đây..."
                    )
                with gr.Column(scale=1):
                    gr.Markdown("**⚙️ Tham số**")
                    sl_max_t = gr.Slider(60, 256, value=200, step=20, label="Max output tokens")
                    sl_min_t = gr.Slider(30, 100, value=60, step=10, label="Min output tokens")
                    sl_beam_t = gr.Slider(2, 8, value=4, step=1, label="Num beams")

            btn_txt = gr.Button("🚀 Tóm tắt", variant="primary", size="large")

            with gr.Row():
                out_preview_t = gr.Textbox(label="📜 Transcript (preview)", lines=5, interactive=False)
                out_kw_t = gr.Textbox(label="🔑 Từ khóa chính", lines=3, interactive=False)

            with gr.Row():
                out_bart_t = gr.Textbox(label="📌 BART Summary (Đề xuất)", lines=8, interactive=False, elem_id="bart-output")
                out_lead3_t = gr.Textbox(label="📋 Lead-3 Baseline", lines=6, interactive=False, elem_id="lead3-output")

            out_stats_t = gr.Textbox(label="📊 Thống kê & Đánh giá", lines=12, interactive=False, elem_id="stats-output")

            btn_txt.click(
                fn=gradio_text,
                inputs=[txt_in, sl_max_t, sl_min_t, sl_beam_t],
                outputs=[out_preview_t, out_kw_t, out_bart_t, out_lead3_t, out_stats_t]
            )

        # Tab 2: Upload File
        with gr.Tab("📁 Upload File (.srt / .txt)"):
            file_in = gr.File(label="Upload file phụ đề hoặc transcript", file_types=[".srt", ".txt", ".vtt"])
            btn_file = gr.Button("🚀 Xử lý & Tóm tắt", variant="primary")

            with gr.Row():
                out_preview_f = gr.Textbox(label="📜 Transcript (preview)", lines=5, interactive=False)
                out_kw_f = gr.Textbox(label="🔑 Từ khóa chính", lines=3, interactive=False)
            with gr.Row():
                out_bart_f = gr.Textbox(label="📌 BART Summary", lines=8, interactive=False, elem_id="bart-output")
                out_lead3_f = gr.Textbox(label="📋 Lead-3 Baseline", lines=6, interactive=False, elem_id="lead3-output")
            out_stats_f = gr.Textbox(label="📊 Thống kê & Đánh giá", lines=12, interactive=False, elem_id="stats-output")

            btn_file.click(
                fn=gradio_srt,
                inputs=[file_in],
                outputs=[out_preview_f, out_kw_f, out_bart_f, out_lead3_f, out_stats_f]
            )

        # Tab 3: Upload Video
        with gr.Tab("🎬 Upload Video (Whisper ASR)"):
            with gr.Row():
                vid_in = gr.Video(label="📹 Upload Video (mp4, mkv, avi)")
                with gr.Column():
                    gr.Markdown("**⚙️ Tham số**")
                    sl_max_v = gr.Slider(60, 256, value=200, step=20, label="Max output tokens")
                    sl_min_v = gr.Slider(30, 100, value=60, step=10, label="Min output tokens")
                    sl_beam_v = gr.Slider(2, 8, value=4, step=1, label="Num beams")
                    dd_whisper = gr.Dropdown(
                        choices=["tiny", "base", "small"],
                        value="base",
                        label="Whisper Model (tiny = nhanh nhất)"
                    )

            btn_vid = gr.Button("🔊 Transcribe → Tóm tắt", variant="primary", size="large")

            with gr.Row():
                out_preview_v = gr.Textbox(label="📜 Transcript (Whisper)", lines=6, interactive=False)
                out_kw_v = gr.Textbox(label="🔑 Từ khóa chính", lines=3, interactive=False)
            with gr.Row():
                out_bart_v = gr.Textbox(label="📌 BART Summary", lines=8, interactive=False, elem_id="bart-output")
                out_lead3_v = gr.Textbox(label="📋 Lead-3 Baseline", lines=6, interactive=False, elem_id="lead3-output")
            out_stats_v = gr.Textbox(label="📊 Thống kê & Đánh giá", lines=12, interactive=False, elem_id="stats-output")

            btn_vid.click(
                fn=gradio_video,
                inputs=[vid_in, sl_max_v, sl_min_v, sl_beam_v, dd_whisper],
                outputs=[out_preview_v, out_kw_v, out_bart_v, out_lead3_v, out_stats_v]
            )

    gr.Markdown("---\n**Mô hình**: BART-large-cnn fine-tuned trên VT-SSum (6.000 mẫu) | ROUGE-1=42.16% | ROUGE-L=24.98% | BERTScore=0.8555")

demo.launch(
    debug=True,
    share=True,
    max_file_size="500mb"
)

/tmp/ipykernel_1348/1775915420.py:29: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="🎥 Video Summarization - Topic 17", theme=gr.themes.Soft(primary_hue="purple"), css=CSS) as demo:
/tmp/ipykernel_1348/1775915420.py:29: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="🎥 Video Summarization - Topic 17", theme=gr.themes.Soft(primary_hue="purple"), css=CSS) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://64545cc375885b817d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


⏳ Loading Whisper base...


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 201MiB/s]


✅ Whisper base loaded!
📦 Video size: 49.4 MB
Detected language: English


 99%|█████████▉| 74092/74642 [00:21<00:00, 3377.38frames/s]


✅ Transcribed 1669 từ (26.6s)
